In [1]:
import asyncio
from clickhouse_demo import ClickHouseDemo
from github_api import GitHubAPI
from github_data_converter import GithubDataConverter


quantile_values = [0.75, 0.85, 0.95, 0.99, 0.999]
click_house_demo = ClickHouseDemo()
clickhouse_data = click_house_demo.get_repos_data("2024-01-01", "2024-04-01", 4500)

github_api = GitHubAPI()
github_api_data = await github_api.get_data(clickhouse_data["repo"])
# df2 = asyncio.run(github_api.get_data(repos))

github_data_converter = GithubDataConverter()
github_data = github_data_converter.merge_data_tables_by_repo(clickhouse_data, github_api_data)
github_data

,pushes,avg_push_size,pull_requests,merged_pull_requests_ratio,issues,closed_issues_ratio,watches,forks,language,license_name,is_deleted_or_private
0,2,1.0,0,0.0,0,0.000000,25,0,None,None,True
1,26,1.0,0,0.0,7,0.428571,11,0,C++,None,False
2,1,1.0,0,0.0,0,0.000000,133,0,None,None,True
3,2,1.0,0,0.0,2,0.000000,20,0,None,None,True
4,0,0.0,0,0.0,0,0.000000,12,2,Python,Other,False
...,...,...,...,...,...,...,...,...,...,...,...
4494,3,1.0,2,0.0,0,0.000000,15,1,Lua,MIT License,False
4495,1,1.0,0,0.0,0,0.000000,10,0,None,None,True
4496,1,1.0,0,0.0,0,0.000000,14,0,None,None,True
4497,0,0.0,0,0.0,0,0.000000,10,0,None,None,True


In [2]:
quantiles = click_house_demo.get_quantiles("2024-01-01",
                             "2024-02-01", 
                             ['forks', 'pushes','avg_push_size', 
                              'pull_requests', 'merged_pull_requests_ratio', 
                              'issues', 'closed_issues_ratio', 'watches'],
                             quantile_values)
quantiles

,forks,pushes,avg_push_size,pull_requests,merged_pull_requests_ratio,issues,closed_issues_ratio,watches
0.750,2.000,7.000,1.103448,6.000,0.000000,4.000,0.500000,2.000
0.850,2.000,11.000,1.666667,12.000,0.111111,8.000,0.666667,3.000
0.950,7.000,31.000,4.500000,31.000,0.500000,22.000,1.000000,9.000
0.990,35.000,110.180,85.707857,94.000,0.571429,75.090,1.000000,47.090
0.999,154.236,718.517,2829.001000,381.652,1.000000,261.899,1.000000,437.944


In [3]:
from pattern_miner import PatternMiner


quantiles_data = github_data_converter.convert_data_to_quantiles(github_data, quantiles)
encoded_data = github_data_converter.encode_quantiles(quantiles_data, quantile_values)
pattern_miner = PatternMiner()
patterns = pattern_miner.mine_patterns(encoded_data, 0.1, 0.1, 1)
patterns

,antecedents,consequents,support,confidence,lift
0,(Мало пушей),(Удален или скрыт),0.3905,0.5170,1.2956
1,(Удален или скрыт),(Мало пушей),0.3905,0.9787,1.2956
2,(Мало размера пушей),(Удален или скрыт),0.3981,0.4391,1.1005
3,(Удален или скрыт),(Мало размера пушей),0.3981,0.9977,1.1005
4,(Мало запросов на слияние),(Удален или скрыт),0.3983,0.4203,1.0534
...,...,...,...,...,...
2867,"(Мало ишью), (Мало форков), (Выше среднего звезд)",(Мало закрытых ишью),0.5978,0.9966,1.0005
2868,"(Мало ишью), (Мало закрытых ишью), (Выше средн...",(Мало форков),0.5978,0.9192,1.0677
2869,"(Мало форков), (Мало закрытых ишью), (Выше сре...",(Мало ишью),0.5978,0.9673,1.0282
2880,"(Мало ишью), (Мало форков), (Много звезд)",(Мало закрытых ишью),0.2061,0.9967,1.0006


In [4]:
patterns.to_excel("data/patterns.xlsx")